# Robodog Control with Hand Pose Workshop 

Welcome! This all-in-one notebook walks you through the complete hand-pose robot controller in three parts:
- **Part 0: Setup** — Install Python **3.11.9**, create a `.venv`, verify your packages, and understand the core concepts.
- **Part 1: Record Poses** — Open a live desktop webcam window to save your 9 personal hand shapes into `gestures.json`.
- **Part 2: Drive the Robot** — Recognize your saved poses live and send movement commands directly to the robot over your local Wi-Fi or `localhost`.

Because Jupyter runs directly on your laptop (instead of a remote cloud server), it can open native OpenCV camera windows and reach `localhost` or `192.168.x.x` robot addresses directly on your network.


## Part 0: Install Python 3.11.9 & Create a Virtual Environment (`venv`)

Follow these steps once on your computer before running the code cells. *(Instructions are written for **Windows** first; macOS/Linux commands are shown right below.)*

### Windows Setup (Step by Step)

1. **Download Python 3.11.9**:
   - Go to [https://www.python.org/downloads/release/python-3119/](https://www.python.org/downloads/release/python-3119/).
   - Scroll down to **Files** and download **Windows installer (64-bit)**.
2. **Install Python 3.11.9**:
   - Double-click the downloaded `.exe` installer.
   - **Important:** At the bottom of the first screen, tick **"Add python.exe to PATH"** (and tick **"Use admin privileges when installing py.exe"** if shown).
   - Click **Install Now**, wait for setup to finish, and click **Close**.
3. **Verify the Python version**:
   - Press the Windows Start key, type `cmd`, and press **Enter** to open **Command Prompt**.
   - Run:
     ```bat
     py -3.11 --version
     ```
   - It should print `Python 3.11.9`.
4. **Create a workshop folder** (and copy `hand_pose_local.ipynb` into it):
   ```bat
   mkdir %USERPROFILE%\robodog-workshop
   cd %USERPROFILE%\robodog-workshop
   ```
5. **Create a virtual environment (`.venv`)**:
   - **What is a `venv`?** Think of a `venv` as a private toolbox folder just for this project. Every Python library we install stays inside `.venv`, so it never breaks or clashes with other Python apps on your laptop.
   ```bat
   py -3.11 -m venv .venv
   ```
6. **Activate the `.venv`**:
   - In **Command Prompt (`cmd`)**:
     ```bat
     .venv\Scripts\activate.bat
     ```
   - Or in **PowerShell**:
     ```powershell
     .venv\Scripts\Activate.ps1
     ```
     *(If PowerShell says running scripts is disabled, run `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` once and press `Y`, then run `.venv\Scripts\Activate.ps1` again.)*
   - When activated, your terminal prompt begins with **`(.venv)`**.
7. **Install the required packages inside `.venv`**:
   ```bat
   python -m pip install --upgrade pip
   pip install jupyter mediapipe opencv-python numpy requests
   ```
   *(We install **`opencv-python`**—not the headless version—so the native desktop webcam window can open.)*
8. **Start Jupyter Notebook**:
   ```bat
   jupyter notebook
   ```
   Your browser will open automatically. Click **`hand_pose_local.ipynb`** to open this notebook.
   *(Next time you reopen the workshop: open `cmd`, `cd %USERPROFILE%\robodog-workshop`, run `.venv\Scripts\activate`, and run `jupyter notebook`.)*

---

### macOS / Linux Setup (Quick Reference)
Install **Python 3.11.9** from [python.org](https://www.python.org/downloads/release/python-3119/) (macOS) or your Linux package manager, copy this notebook into a folder, then run:
```bash
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install jupyter mediapipe opencv-python numpy requests
jupyter notebook
```

---

### How to Run Cells Here & Webcam Tip
- **Running a cell:** Click inside a grey code box and press **`Shift + Enter`** (or click the **▶ Run** button in the top toolbar).
- While a cell is running, the prompt beside it shows **`[*]:`**; wait until it changes to a number like **`[1]:`** before running the next cell.
- **One app per webcam:** Close Zoom, Teams, Camera, or other browser video apps before running Part 1 or Part 2—only one program can use your camera at a time.

Run the **Environment Check** cell below to verify your Python 3.11 `.venv` and installed packages.


In [11]:
import importlib
import sys

print("Python version   :", sys.version.split()[0])
print("Python executable:", sys.executable)

if sys.version_info[:2] != (3, 11):
    print("WARNING: Expected Python 3.11.x (recommended 3.11.9). Recreate .venv with py -3.11 -m venv .venv.")

in_venv = sys.prefix != sys.base_prefix
if not in_venv:
    print("WARNING: Not running inside a virtual environment (.venv). Activate .venv before launching jupyter notebook.")
else:
    print("Virtual env active:", sys.prefix)

missing = []
for pkg in ("cv2", "mediapipe", "numpy", "requests"):
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "installed")
        print(f"  [OK] {pkg} ({ver})")
    except Exception as exc:
        missing.append(pkg)
        print(f"  [MISSING] {pkg}: {exc}")

if missing:
    print("\nInstall missing packages inside your activated .venv with:")
    print("  pip install jupyter mediapipe opencv-python numpy requests")
else:
    print("\nAll required packages are ready!")

Python version   : 3.11.9
Python executable: /mnt/volume-ssd-ubuntu-2/projects/cenim/robodog-pose-gesture/notebooks/local/.venv/bin/python
Virtual env active: /mnt/volume-ssd-ubuntu-2/projects/cenim/robodog-pose-gesture/notebooks/local/.venv
  [OK] cv2 (5.1.0)
  [OK] mediapipe (1.2.3)
  [OK] numpy (2.4.6)
  [OK] requests (2.34.2)

All required packages are ready!


## Key Concepts

Before we configure and run the robot controller, here is what every moving part means in plain language:

### 1. What Is a "Pose Class"?
In machine learning, a **class** is just a named category—like labeling photo folders `"cat"` and `"dog"`. Here, each class is one specific hand shape mapped to one robot action. We store up to **9 classes** in `gestures.json`:

| Class Name | What the Robot Does | Hand Shape Idea (Pick Clear, Different Poses!) |
|---|---|---|
| `forward` | Walks forward | Open flat palm or 1 finger pointing up |
| `backward` | Walks backward | Thumbs down or downward palm |
| `left` | Sidesteps left | Thumb pointing left / L-shape |
| `right` | Sidesteps right | Pinky/hand pointing right |
| `turn_left` | Rotates counter-clockwise | Peace sign (2 fingers up) |
| `turn_right` | Rotates clockwise | 3 fingers up |
| `sit` | Robot sits / lies down | OK sign (thumb + index circle) |
| `stand` | Robot stands up | Thumbs up |
| `stop` | Safety stop (all speeds set to `0`) | Closed fist or flat stop hand |

### 2. What Is an API?
An **API** *(Application Programming Interface)* works like a **receptionist or waiter** on the robot. Instead of wiring Python directly into the robot's leg motors, we send polite command messages to the robot's HTTP server (`/api/topics/publish`), and the server forwards them to the robot.

### 3. What Is an HTTP `POST` Request & JSON Payload?
Just like your web browser loads web pages over `http://`, Python sends robot orders using an **HTTP `POST` request** carrying a **JSON payload** (data inside `{...}` curly braces):
```json
{"topic": "/cmd_vel", "type": "geometry_msgs/Twist", "payload": {"linear": {"x": 0.2, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}}
```
- **`topic`**: Which channel on the robot to publish to (`/cmd_vel` for movement speeds, `/simple_cmd` for `sit`/`stand`).
- **`type`**: Message format expected by the robot (`geometry_msgs/Twist` or `message_transformer/SimpleCMD`).
- **`payload`**: The speed numbers or posture command code.

### 4. What Are an IP Address, Port, and `localhost`?
- **IP Address** (e.g., `192.168.1.100`): The robot's "street address" on your Wi-Fi network so commands reach the right machine.
- **Port** (e.g., `:3000`): The specific "door number" on that machine where the API server is listening.
- **`localhost`**: A special nickname meaning **"this exact computer I am typing on right now"** (used if a robot bridge or simulator runs locally on your laptop).

### 5. What Is CORS and Why Don't We Need Tricks Here?
- **CORS** *(Cross-Origin Resource Sharing)* is a **web browser security rule** that restricts JavaScript web pages from calling local servers unless special permission headers are configured.
- In this local Jupyter notebook, every HTTP `POST` request is sent **directly from Python** (`requests.Session().post(..., json=body, headers={"Accept": "application/json"})`). Because Python is not a browser webpage, browser CORS rules do not restrict it, and the request is a clean, standard `Content-Type: application/json` HTTP request!

### 6. How Your Local Notebook Talks to the Robot Over Wi-Fi
Because Jupyter is running right inside `.venv` on your laptop, Python sits directly on your workshop Wi-Fi/LAN—so both `http://localhost:3000/...` and `http://192.168.x.x:3000/...` work straight away with no tunnels or browser settings needed.


## Block 1: Setup Model, API Configuration & Helpers

### Step 1.1: Download MediaPipe Hand Landmarker Model
MediaPipe uses a lightweight AI model (`hand_landmarker.task`) that locates **21 key points (dots)** on your wrist, knuckles, and fingertips in each video frame. Run this cell to download `hand_landmarker.task` into the current folder (about 7.5 MB; skipped automatically if already present).


In [12]:
import urllib.request
from pathlib import Path

MODEL_PATH = Path("hand_landmarker.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/hand_landmarker.task"
)
if not MODEL_PATH.exists():
    print("Downloading hand_landmarker.task...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Model ready:", MODEL_PATH.resolve())

Model ready: /mnt/volume-ssd-ubuntu-2/projects/cenim/robodog-pose-gesture/notebooks/local/hand_landmarker.task


### Step 1.2: Configure `API_URL`, Camera Index, Payloads & Tolerance

#### Setup Checklist
1. Connect your laptop to the **workshop Wi-Fi** that the robot dog is on.
2. Ask your instructor for the robot's **IP address** (for example `192.168.1.100`), or keep `localhost` if the bridge runs on your own laptop.
3. In the code cell below, edit **only the host part** of `API_URL` (between `http://` and `:3000`):
   - `API_URL = "http://192.168.1.100:3000/api/topics/publish"` *(robot on Wi-Fi)*
   - `API_URL = "http://localhost:3000/api/topics/publish"` *(bridge on your laptop)*
   - Keep `http://`, `:3000`, and `/api/topics/publish` exact. *(No browser settings are needed locally!)*

#### What These Settings Control
- **`CAMERA_INDEX` (`0`)**: Built-in laptop webcam is usually `0`; if you plugged in an external USB webcam, try `1`.
- **`TOLERANCE` (`0.15`)**: Strictness threshold for matching a hand pose. Lower (`0.10`) = stricter; higher (`0.20`) = easier to trigger.
- **`CLASS_PAYLOADS` (`/cmd_vel` Speeds)**:
  | Parameter | Controls | Positive (`+`) | Negative (`-`) |
  |---|---|---|---|
  | `lx` (`linear.x`) | Forward / backward speed | `forward` (`0.2`) | `backward` (`-0.2`) |
  | `ly` (`linear.y`) | Sideways step speed | `left` (`0.1`) | `right` (`-0.1`) |
  | `az` (`angular.z`) | Turning rotation speed | `turn_left` (`0.3`) | `turn_right` (`-0.3`) |
- **`SIMPLE_CMD_PAYLOADS` (`/simple_cmd` Codes)**: `stand` sends `0x21010202` and `sit` sends `0x21010203` once whenever you switch into that pose.
- **`ZERO_TWIST` Safety Brake**: Whenever no hand is visible or your pose says `Unknown`, the detector automatically publishes `0.0` for all speeds so the robot stops safely.


In [18]:
# === API Endpoint, Camera & Robot Payload Configuration ===
API_URL = "http://127.0.0.1:3000/api/topics/publish"  # <-- Replace 'localhost' with the robot's Wi-Fi IP if needed (e.g. 192.168.1.100)
TOPIC = "/cmd_vel"                                    # Robot topic for continuous walking/turning speeds
MSG_TYPE = "geometry_msgs/Twist"
SIMPLE_CMD_TOPIC = "/simple_cmd"                      # Robot topic for one-shot posture commands (sit/stand)
SIMPLE_CMD_TYPE = "message_transformer/SimpleCMD"
STAND_CMD_CODE = 0x21010202  # 553714178 (stand up)
SIT_CMD_CODE = 0x21010203    # 553714179 (sit / lie down)

PUBLISH_HZ = 10
PUBLISH_INTERVAL = 1.0 / PUBLISH_HZ  # 0.1s between velocity commands (10 times per second)
HTTP_TIMEOUT = 0.3                   # Lower timeout (e.g. 0.1) keeps webcam video smoother when the robot is offline

CAMERA_INDEX = 0                     # 0 = default laptop webcam; try 1 for an external USB camera
GESTURES_PATH = Path("gestures.json")  # Stored in the same folder as this notebook
TOLERANCE = 0.15                     # Max average landmark difference allowed to match a pose

POSE_CLASSES = (
    "forward",
    "backward",
    "left",
    "right",
    "turn_left",
    "turn_right",
    "sit",
    "stand",
    "stop",
)


def make_twist(lx: float = 0.0, ly: float = 0.0, lz: float = 0.0, az: float = 0.0) -> dict:
    return {
        "linear": {"x": lx, "y": ly, "z": lz},
        "angular": {"x": 0.0, "y": 0.0, "z": az},
    }


def make_simple_cmd(cmd_code: int, cmd_value: int = 0, cmd_type: int = 0) -> dict:
    return {"cmd_code": cmd_code, "cmd_value": cmd_value, "type": cmd_type}


CLASS_PAYLOADS: dict[str, dict] = {
    "forward": make_twist(lx=0.2),
    "backward": make_twist(lx=-0.2),
    "left": make_twist(ly=0.1),
    "right": make_twist(ly=-0.1),
    "turn_left": make_twist(az=0.3),
    "turn_right": make_twist(az=-0.3),
    "stop": make_twist(),
}
SIMPLE_CMD_PAYLOADS: dict[str, dict] = {
    "sit": make_simple_cmd(SIT_CMD_CODE),
    "stand": make_simple_cmd(STAND_CMD_CODE),
}
ZERO_TWIST = make_twist()

print("Configuration ready. Robot API_URL:", API_URL)

Configuration ready. Robot API_URL: http://127.0.0.1:3000/api/topics/publish


### Step 1.3: Load Landmark Math, Skeleton Drawing, Recorder UI & HTTP Helpers

Run this cell to load all helper functions used in Part 1 and Part 2:
- **`draw_hand_skeleton()`**: Draws the 21 hand dots and connecting bones on your camera feed (supports both older `0.10.x` and newer `1.2.x` MediaPipe versions).
- **`normalize_landmarks()`**: Centers landmarks on the wrist (`#0`) and scales by hand size so distance from the camera does not break recognition.
- **`load_gestures()` / `save_gesture()`**: Reads and writes `gestures.json` right next to this notebook.
- **`classify_gesture()`**: Finds the closest saved pose within `TOLERANCE` and returns `(label, distance, closest_name)`.
- **`create_video_landmarker()`**: Creates a MediaPipe `HandLandmarker` in live `VIDEO` tracking mode.
- **`build_publish_body()` / `publish_command()`**: Uses a persistent `requests.Session()` to send clean HTTP `POST` JSON commands (`Content-Type: application/json`, `Accept: application/json`) directly from Python to `API_URL`.
- **`draw_debug_ui()`**: Renders clickable class buttons and **SAVE POSE (S)** inside the Part 1 OpenCV window.


In [25]:
import json
import time
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import requests
from mediapipe.tasks.python import vision

try:
    BaseOptions = mp.tasks.BaseOptions
except AttributeError:
    from mediapipe.tasks.python import core
    BaseOptions = core.base_options.BaseOptions

HAND_CONNECTIONS = (
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17),
)

_http = requests.Session()


def draw_hand_skeleton(frame: np.ndarray, hand_landmarks) -> None:
    """Draw MediaPipe hand landmarks and bones onto frame."""
    if (
        hasattr(vision, "drawing_utils")
        and hasattr(vision, "drawing_styles")
        and hasattr(vision, "HandLandmarksConnections")
    ):
        vision.drawing_utils.draw_landmarks(
            frame,
            hand_landmarks,
            vision.HandLandmarksConnections.HAND_CONNECTIONS,
            vision.drawing_styles.get_default_hand_landmarks_style(),
            vision.drawing_styles.get_default_hand_connections_style(),
        )
        return
    h, w = frame.shape[:2]
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 220, 0), 2, cv2.LINE_AA)
    for x, y in pts:
        cv2.circle(frame, (x, y), 4, (0, 80, 255), -1, cv2.LINE_AA)


def normalize_landmarks(hand_landmarks) -> np.ndarray:
    """Center landmarks at the wrist (0) and scale by max distance from wrist."""
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks], dtype=np.float32)
    centered = coords - coords[0]
    scale = float(np.max(np.linalg.norm(centered, axis=1)))
    return centered / scale if scale > 1e-6 else centered


def load_gestures(path: Path = GESTURES_PATH) -> dict[str, np.ndarray]:
    """Load saved gesture templates {name: (21, 3) ndarray} from JSON."""
    p = Path(path)
    if not p.exists():
        return {}
    raw = json.loads(p.read_text(encoding="utf-8"))
    return {name: np.asarray(coords, dtype=np.float32) for name, coords in raw.items()}


def save_gesture(path: Path, name: str, coords: np.ndarray) -> dict[str, np.ndarray]:
    """Save or overwrite a gesture template in gestures.json."""
    p = Path(path)
    gestures = load_gestures(p)
    gestures[name] = coords.astype(np.float32)
    p.parent.mkdir(parents=True, exist_ok=True)
    serializable = {k: np.round(v, 6).tolist() for k, v in gestures.items()}
    p.write_text(json.dumps(serializable, indent=2) + "\n", encoding="utf-8")
    return gestures


def classify_gesture(
    live_coords: np.ndarray,
    gestures: dict[str, np.ndarray],
    tolerance: float = TOLERANCE,
) -> tuple[str, float | None, str | None]:
    """Match normalized landmarks against saved gestures; returns (label, dist, closest)."""
    if not gestures:
        return "No gestures saved", None, None
    best_name: str | None = None
    best_dist = float("inf")
    for name, ref_coords in gestures.items():
        if ref_coords.shape != live_coords.shape:
            continue
        dist = float(np.mean(np.linalg.norm(live_coords - ref_coords, axis=1)))
        if dist < best_dist:
            best_dist = dist
            best_name = name
    if best_name is None:
        return "No gestures saved", None, None
    if best_dist <= tolerance:
        return best_name, best_dist, best_name
    return "Unknown", best_dist, best_name


def create_video_landmarker(num_hands: int = 1) -> vision.HandLandmarker:
    """Create a MediaPipe HandLandmarker configured for live video frames."""
    options = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=vision.RunningMode.VIDEO,
        num_hands=num_hands,
    )
    return vision.HandLandmarker.create_from_options(options)


def build_publish_body(class_name: str | None) -> tuple[dict, str]:
    """Build REST JSON body for /simple_cmd (sit/stand) or /cmd_vel (movement/zero)."""
    if class_name in SIMPLE_CMD_PAYLOADS:
        topic, msg_type = SIMPLE_CMD_TOPIC, SIMPLE_CMD_TYPE
        payload, label = SIMPLE_CMD_PAYLOADS[class_name], class_name
    else:
        topic, msg_type = TOPIC, MSG_TYPE
        is_known = class_name in CLASS_PAYLOADS
        payload = CLASS_PAYLOADS[class_name] if is_known else ZERO_TWIST
        label = class_name if is_known else "zero"
    return {"topic": topic, "type": msg_type, "payload": payload}, label


def publish_command(class_name: str | None) -> str:
    """Send HTTP POST JSON directly from Python to API_URL."""
    body, label = build_publish_body(class_name)
    try:
        resp = _http.post(
            API_URL,
            json=body,
            headers={"Accept": "application/json"},
            timeout=HTTP_TIMEOUT,
        )
        return f"HTTP {resp.status_code}: {label}"
    except requests.RequestException as exc:
        return f"HTTP err ({label}): {exc}"


def _process_video_frame(
    frame: np.ndarray,
    landmarker: vision.HandLandmarker,
    timestamp_ms: int,
    gestures: dict[str, np.ndarray],
    tolerance: float,
    debug: bool = False,
) -> tuple[np.ndarray | None, tuple[str, float | None, str | None] | None]:
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect_for_video(mp_image, timestamp_ms)

    primary_coords: np.ndarray | None = None
    primary_match: tuple[str, float | None, str | None] | None = None
    h, w = frame.shape[:2]

    for idx, hand_lms in enumerate(result.hand_landmarks):
        draw_hand_skeleton(frame, hand_lms)
        coords = normalize_landmarks(hand_lms)
        match = classify_gesture(coords, gestures, tolerance)
        label, dist, closest = match

        if idx == 0:
            primary_coords = coords
            primary_match = match

        wrist_x = int(hand_lms[0].x * w)
        wrist_y = max(30, int(hand_lms[0].y * h) - 15)
        is_known = dist is not None and label not in ("Unknown", "No gestures saved")
        color = (0, 220, 0) if is_known else (0, 180, 255)
        text = f"{label} ({closest}: {dist:.3f})" if (debug and dist is not None) else label
        cv2.putText(
            frame,
            text,
            (max(10, wrist_x - 40), wrist_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2,
            cv2.LINE_AA,
        )

    banner = "Hand: none"
    banner_color = (180, 180, 180)
    if primary_match is not None:
        label, dist, _ = primary_match
        banner = f"Gesture: {label}"
        if dist is not None:
            banner += f" (err={dist:.3f}, tol={tolerance:.2f})"
        banner_color = (0, 220, 0) if label not in ("Unknown", "No gestures saved") else (0, 180, 255)

    cv2.rectangle(frame, (10, 10), (460, 50), (25, 25, 25), -1)
    cv2.putText(frame, banner, (20, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.7, banner_color, 2, cv2.LINE_AA)
    return primary_coords, primary_match


def compute_ui_layout(frame_shape: tuple[int, ...]) -> tuple[tuple[int, int, int, int], list[tuple[str, tuple[int, int, int, int]]]]:
    """Compute button rectangles for SAVE POSE and each selectable class."""
    h, w = frame_shape[:2]
    x2 = w - 10
    x1 = max(10, x2 - 165)
    save_rect = (x1, 10, x2, 50)

    gap = 4
    btn_h = max(22, min(32, (h - 95) // len(POSE_CLASSES) - gap))
    class_rects: list[tuple[str, tuple[int, int, int, int]]] = []
    y = 60
    for cls_name in POSE_CLASSES:
        class_rects.append((cls_name, (x1, y, x2, y + btn_h)))
        y += btn_h + gap
    return save_rect, class_rects


def _point_in_rect(x: int, y: int, rect: tuple[int, int, int, int]) -> bool:
    x1, y1, x2, y2 = rect
    return x1 <= x <= x2 and y1 <= y <= y2


def draw_debug_ui(
    frame: np.ndarray,
    gestures: dict[str, np.ndarray],
    selected_idx: int,
    tolerance: float,
    status_msg: str,
) -> tuple[tuple[int, int, int, int], list[tuple[str, tuple[int, int, int, int]]]]:
    """Render debug info, SAVE button, and selectable class buttons."""
    h, _ = frame.shape[:2]
    save_rect, class_rects = compute_ui_layout(frame.shape)
    selected_class = POSE_CLASSES[selected_idx]

    sx1, sy1, sx2, sy2 = save_rect
    cv2.rectangle(frame, (sx1, sy1), (sx2, sy2), (40, 140, 240), -1)
    cv2.rectangle(frame, (sx1, sy1), (sx2, sy2), (255, 255, 255), 2)
    cv2.putText(frame, "SAVE POSE (S)", (sx1 + 12, sy1 + 27), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2, cv2.LINE_AA)

    for idx, (cls_name, (cx1, cy1, cx2, cy2)) in enumerate(class_rects):
        is_selected = idx == selected_idx
        is_saved = cls_name in gestures
        fill = (180, 110, 20) if is_selected else (40, 40, 40)
        border = (0, 255, 255) if is_selected else ((0, 200, 0) if is_saved else (140, 140, 140))
        cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), fill, -1)
        cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), border, 2 if is_selected else 1)
        tag = "*" if is_saved else " "
        cv2.putText(
            frame,
            f"{idx + 1}:{cls_name} {tag}",
            (cx1 + 8, cy1 + (cy2 - cy1) // 2 + 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.48,
            (255, 255, 255),
            1,
            cv2.LINE_AA,
        )

    cv2.putText(
        frame,
        f"Target: {selected_class} | Saved: {len(gestures)}/{len(POSE_CLASSES)}",
        (12, 74),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.58,
        (230, 230, 230),
        1,
        cv2.LINE_AA,
    )
    if status_msg:
        cv2.putText(frame, status_msg, (12, 102), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 255, 255), 2, cv2.LINE_AA)

    cv2.putText(
        frame,
        f"Tol: {tolerance:.2f} (+/-) | 1-9/Click: Select | S/Save: Store | Q/ESC: Quit",
        (12, h - 12),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (220, 220, 220),
        1,
        cv2.LINE_AA,
    )
    return save_rect, class_rects


print("Loaded gestures:", list(load_gestures().keys()))

Loaded gestures: ['forward']


## Part 1: Record Your Hand Poses (`gestures.json`)

Follow these steps to capture your personal hand shapes into `gestures.json` (stored right next to this notebook—no manual file upload/download needed!):

1. **Launch the recorder**: Run the cell below (`Shift + Enter`). While the camera window is open, the Jupyter cell beside it shows **`[*]:`**—that is completely normal!
2. **Find and focus the window**: A desktop window titled **`Hand Pose Recorder`** will open. *(On Windows/macOS it can sometimes open behind your browser—check your taskbar and click inside the window so keyboard keys work.)*
3. **Select a class**: Click a class button on the right panel, or press keys **`1`–`9`** (or **`Tab`**):
   - **Blue fill + yellow border**: Currently selected target class.
   - **Green border + `*`**: Class already has a saved pose in `gestures.json`.
4. **Save your pose**: Hold your hand clearly in front of the webcam until the skeleton lines appear, then click **`SAVE POSE (S)`** or press **`S`**.
5. **Test & tune**: Try making each saved pose and check the top banner (`Gesture: forward (err=0.042, tol=0.15)`). You can nudge tolerance live with **`+`** / **`-`** keys, and **overwrite** any class anytime by selecting it again and pressing **`S`**.
6. **Quit when finished**: Press **`Q`** or **`ESC`** inside the `Hand Pose Recorder` window to close it cleanly before moving to Part 2.

### Part 1 Troubleshooting
- **Window not visible?** Check your OS taskbar/Dock for **`Hand Pose Recorder`**—it often spawns behind your browser.
- **Camera won't open or `Could not open camera index 0`?** Close any other video app using the webcam (Zoom/Teams/browser), or change `CAMERA_INDEX = 1` in Step 1.2 and re-run Step 1.2.
- **Always shows `Unknown`?** Make sure all 9 classes have distinct hand shapes, hold your whole hand in view, or press `+` to raise `tolerance` slightly.


In [20]:
def register_poses(camera_index: int = CAMERA_INDEX, tolerance: float = TOLERANCE) -> None:
    """Open the native OpenCV pose registration and debug window."""
    win_name = "Hand Pose Recorder"
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open camera index {camera_index}")

    gestures = load_gestures(GESTURES_PATH)
    selected_idx = 0
    save_requested = False
    save_rect = (0, 0, 0, 0)
    class_rects: list[tuple[str, tuple[int, int, int, int]]] = []

    def on_mouse(event: int, x: int, y: int, _flags: int, _param) -> None:
        nonlocal selected_idx, save_requested
        if event != cv2.EVENT_LBUTTONDOWN:
            return
        if _point_in_rect(x, y, save_rect):
            save_requested = True
            return
        for idx, (_cls_name, rect) in enumerate(class_rects):
            if _point_in_rect(x, y, rect):
                selected_idx = idx
                return

    cv2.namedWindow(win_name)
    cv2.setMouseCallback(win_name, on_mouse)

    status_msg = ""
    status_until = 0.0
    last_ts = 0

    try:
        with create_video_landmarker(num_hands=1) as landmarker:
            while True:
                now = time.monotonic()
                if now > status_until:
                    status_msg = ""

                ok, frame = cap.read()
                if not ok:
                    break
                frame = cv2.flip(frame, 1)
                last_ts = max(last_ts + 1, int(now * 1000))
                latest_coords, _ = _process_video_frame(
                    frame, landmarker, last_ts, gestures, tolerance, debug=True
                )

                save_rect, class_rects = draw_debug_ui(
                    frame, gestures, selected_idx, tolerance, status_msg
                )
                cv2.imshow(win_name, frame)

                if save_requested:
                    save_requested = False
                    target_class = POSE_CLASSES[selected_idx]
                    if latest_coords is None:
                        status_msg = "No hand detected - show hand before saving!"
                    else:
                        gestures = save_gesture(GESTURES_PATH, target_class, latest_coords)
                        status_msg = f"Saved '{target_class}' -> {GESTURES_PATH.name}"
                    status_until = now + 2.5

                key = cv2.waitKey(1) & 0xFF
                if key == 255:
                    continue
                if key in (ord("q"), ord("Q"), 27):
                    break
                if key in (ord("s"), ord("S")):
                    save_requested = True
                elif ord("1") <= key <= ord(str(len(POSE_CLASSES))):
                    selected_idx = key - ord("1")
                elif key == 9:
                    selected_idx = (selected_idx + 1) % len(POSE_CLASSES)
                elif key in (ord("+"), ord("=")):
                    tolerance = min(1.0, round(tolerance + 0.01, 2))
                elif key in (ord("-"), ord("_")):
                    tolerance = max(0.01, round(tolerance - 0.01, 2))
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Saved classes:", list(load_gestures(GESTURES_PATH).keys()))


register_poses()

INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x753ad4610c10, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x753ad40be650, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.


Saved classes: ['forward']


INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x753ad4610c10, name=CpuAccelerator
INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x753ad40be650, name=CpuAccelerator


## Part 2: Live Hand Pose Detection & Robot Control

Now run the live robot controller using the `gestures.json` you just saved!

### What Happens While Running
1. A window titled **`Robodog Live Control`** opens (check your taskbar and click it to focus if needed).
2. Each frame detects your hand, matches it against `gestures.json` (and hot-reloads `gestures.json` automatically if modified), and sends HTTP `POST` JSON requests from Python directly to `API_URL`:
   - **Movement poses** (`forward`, `backward`, `left`, `right`, `turn_left`, `turn_right`, `stop`) publish `geometry_msgs/Twist` to `/cmd_vel` **10 times per second**.
   - **Posture poses** (`sit`, `stand`) publish `message_transformer/SimpleCMD` to `/simple_cmd` **once** when switching into `sit` or `stand`.
   - **Safety Brake (`ZERO_TWIST`)**: When your hand leaves the view or reads `Unknown`, Python publishes all `0.0` speeds so the robot stops.
3. **Reading the Window Overlay**:
   - **Top banner**: `Gesture: forward (err=0.054, tol=0.15)` shows the matched class and error score.
   - **Bottom status line**: `Loaded: 9 gestures | HTTP 200: forward | Q/ESC: quit` shows how many classes are loaded and the live HTTP status from the robot.
4. **Stopping cleanly**: Press **`Q`** or **`ESC`** inside `Robodog Live Control` to exit and release your webcam. *(Tip: Show your `stop` pose or lower your hand first before pressing `Q` so the robot receives a `0.0` stop command before the window closes!)*

### Part 2 Troubleshooting
- **`Loaded: 0 gestures`**: Re-run Part 1 to save poses into `gestures.json` in this same folder.
- **Always shows `Unknown`**: Hold your pose steadily in frame, or slightly increase `TOLERANCE` in Step 1.2 and re-run Step 1.2 + Part 2.
- **`HTTP err ... ConnectTimeout` / `ConnectionError`**:
  1. Check that the robot dog (and its REST API bridge on port `3000`) is powered on.
  2. Check that your laptop is connected to the **same Wi-Fi** as the robot.
  3. Double-check the IP and port in `API_URL` (Step 1.2), and remember to **re-run the Step 1.2 code cell** after editing `API_URL`!
- **Camera error?** Close the Part 1 window (`Q`/`ESC`) or other camera apps first, or try `CAMERA_INDEX = 1` in Step 1.2.


In [26]:
def run_detection(
    camera_index: int = CAMERA_INDEX,
    tolerance: float = TOLERANCE,
    publish_interval: float = PUBLISH_INTERVAL,
) -> None:
    """Run live webcam gesture classification and direct local REST publishing."""
    win_name = "Robodog Live Control"
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open camera index {camera_index}")

    gestures = load_gestures(GESTURES_PATH)
    last_mtime = GESTURES_PATH.stat().st_mtime if GESTURES_PATH.exists() else None
    last_ts = 0
    last_pub_label: str | None = None
    last_pub_time = 0.0
    api_status = "API: idle"
    print(API_URL, "->", TOPIC, MSG_TYPE, "|", SIMPLE_CMD_TOPIC, SIMPLE_CMD_TYPE)

    try:
        with create_video_landmarker(num_hands=1) as landmarker:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break

                if GESTURES_PATH.exists():
                    mtime = GESTURES_PATH.stat().st_mtime
                    if mtime != last_mtime:
                        gestures = load_gestures(GESTURES_PATH)
                        last_mtime = mtime

                frame = cv2.flip(frame, 1)
                now = time.monotonic()
                last_ts = max(last_ts + 1, int(now * 1000))
                _, primary_match = _process_video_frame(
                    frame, landmarker, last_ts, gestures, tolerance, debug=False
                )

                detected = primary_match[0] if primary_match is not None else None
                is_simple = detected in SIMPLE_CMD_PAYLOADS
                active_class = detected if (is_simple or detected in CLASS_PAYLOADS) else None
                should_publish = (
                    (active_class != last_pub_label)
                    if is_simple
                    else (active_class != last_pub_label or now - last_pub_time >= publish_interval)
                )
                if should_publish:
                    api_status = publish_command(active_class)
                    last_pub_label = active_class
                    last_pub_time = now

                cv2.putText(
                    frame,
                    f"Loaded: {len(gestures)} gestures | {api_status[:48]} | Q/ESC: quit",
                    (15, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (220, 220, 220),
                    1,
                    cv2.LINE_AA,
                )
                cv2.imshow(win_name, frame)

                key = cv2.waitKey(1) & 0xFF
                if key in (ord("q"), ord("Q"), 27):
                    break
    finally:
        cap.release()
        cv2.destroyAllWindows()


run_detection()

http://127.0.0.1:3000/api/topics/publish -> /cmd_vel geometry_msgs/Twist | /simple_cmd message_transformer/SimpleCMD


INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x753a800763b0, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
INFO: [environment.cc:36] Creating LiteRT environment with options
INFO: [accelerator_registry.cc:54] RegisterAccelerator: ptr=0x753a7c003b20, name=CpuAccelerator
INFO: [cpu_registry.cc:75] XNNPACK CPU accelerator registered.
INFO: [compiled_model.cc:1022] Flatbuffer model initialized directly from incoming litert model.
INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x753a800763b0, name=CpuAccelerator
INFO: [accelerator_registry.cc:43] DestroyAccelerator: ptr=0x753a7c003b20, name=CpuAccelerator
